In [ ]:
from scvi.train import Trainer
from scvi.model import TOTALVI
import scanpy as sc
from matplotlib_inline.config import InlineBackend
from sklearn.cluster import KMeans
from sklearn.metrics.cluster import adjusted_rand_score
from sklearn.metrics.cluster import normalized_mutual_info_score
import anndata as ad
import scvi
import matplotlib.pyplot as plt 
import sys
sys.path.append("/multiHIVE/src")
from src.model import multiHIVE
import pandas as pd

/home/anirudhn/anaconda3/envs/cloned_env/lib/python3.9/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: libtorch_cuda_cu.so: cannot open shared object file: No such file or directory
  warn(f"Failed to load image Python extension: {e}")
Global seed set to 0


In [ ]:
adata = sc.read('/Data/Biological_Analysis/breast_cancer_cite.h5ad')
adata.layers['counts'] = adata.X.copy()
adata.var_names_make_unique()
adata.obsm['protein_counts'] = adata.obsm['protein_counts'].toarray()
import numpy as np


sc.pp.filter_cells(adata, min_genes=200)
sc.pp.filter_genes(adata, min_cells=3)
sc.pp.highly_variable_genes(
    adata,
    n_top_genes=2000,
    flavor="seurat_v3",
    batch_key="batch",
    subset=True,
    layer="counts"
)


multiHIVE.setup_anndata(
    adata,
    layer="counts",
    batch_key="batch",
    protein_expression_obsm_key="protein_counts"
)
vae = multiHIVE(
    adata, latent_distribution="normal", override_missing_proteins=True, kl_dot_product = True
    n_genes=adata.shape[1],
    n_regions=0,
    n_proteins=adata.obsm["protein_counts"].shape[1],)
vae.train()
vae.get_latent_representation()

In [ ]:
vae.save("./outputs/saved_model/")

In [21]:
protein_names = pd.read_csv("/home/anirudhn/notebooks/breast_cancer_protein_names.txt", sep='\t', header=None)[0].values
adata.obsm['protein_counts'] = pd.DataFrame(adata.obsm['protein_counts'], index = adata.obs_names, columns=protein_names)

In [25]:
generated_data = vae.posterior_predictive_sample(adata, swap_latent=False)
rna_sample = pd.DataFrame(generated_data[:,:2000], index= adata.obs_names, columns = adata.var_names)
proteins_sample = pd.DataFrame(generated_data[:,2000:],  index= adata.obs_names, columns = protein_names)
adata.obsm['RNA_Z1_denoised'] = rna_sample
adata.obsm['protein_Z1_denoised'] = proteins_sample

generated_data = vae.posterior_predictive_sample(adata, swap_latent=True)
rna_sample = pd.DataFrame(generated_data[:,:2000], index= adata.obs_names, columns = adata.var_names)
proteins_sample = pd.DataFrame(generated_data[:,2000:],  index= adata.obs_names, columns = protein_names)
adata.obsm['RNA_Z2_denoised'] = rna_sample
adata.obsm['protein_Z2_denoised'] = proteins_sample

In [ ]:
# adata.write("./outputs/breast_cancer_cite_HierarVI.h5ad")
np.save("./outputs/RNA_Z1_denoised.npy", adata.obsm['RNA_Z1_denoised'])
np.save("./outputs/RNA_Z2_denoised.npy", adata.obsm['RNA_Z2_denoised'])
np.save("./outputs/Z_multiHIVE.npy", adata.obsm['Z_mulitiHIVE'])

In [ ]:
# adata = sc.read_h5ad("./outputs/breast_cancer_cite_HierarVI.h5ad")